## Data validation
- Validate data -> Save at validation folder
    - Data Type not match -> set schema table at config file and prepare init table
    - Not null columns (key) -> set not null at config file
    - Key duplicate -> set key at config file
    - Row duplicate -> set row uniqueness validation in config file

In [6]:
import os
os.chdir("../")

In [7]:
os.getcwd()

'/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction'

### Import library

In [2]:
# import necessary libraries
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from dataclasses import dataclass
from datetime import datetime
import pandera.pandas as pa
from pandera import Column, DataFrameSchema

# import project modules
from src.churn_prediction.logger import logger
from src.churn_prediction.pydantic.data_validation_config import DataValidationConfig
from src.churn_prediction.pydantic.pipeline_config import PipelineConfig
from src.churn_prediction.utils.common import load_single_config, generate_sk_key
from src.churn_prediction.utils.loaders import load_data
from src.churn_prediction.utils.writers import save_data

ModuleNotFoundError: No module named 'src'

In [38]:
admin_group_df = load_data(source="data/user_coop_anonymized.csv", delimiter="|")
admin_group_df['admin_id'] = admin_group_df['admin_id'].astype(str)

[ 2025-11-08 13:01:21 ] | churn_prediction | INFO     | loaders.py:load_data:343 | Auto-detecting format for local file: data/user_coop_anonymized.csv
[ 2025-11-08 13:01:21 ] | churn_prediction | INFO     | loaders.py:load_csv:239 | Loading CSV from local: data/user_coop_anonymized.csv
[ 2025-11-08 13:01:21 ] | churn_prediction | INFO     | loaders.py:load_csv:241 | ✓ Successfully loaded 199160 rows


KeyError: 'admin_id'

In [ ]:
save_data(admin_group_df, "data/raw/customer_profile.parquet")

In [53]:
from pyspark.sql import SparkSession
import numpy as np
import pandas as pd

# Initialize Spark session
spark = SparkSession.builder.appName("Example").getOrCreate()

# Your sample data
sample_data = {
    'user_id': ['2', '2', np.nan, '4', '4'],
    'first_name': ['John', 'Jane', 'Bob', 'Alice', 'Alice'],
    'last_name': ['Doe', 'Smith', 'Johnson', 'Brown', 'Brown'],
    'activated': ['True', 'True', 'False', 'True', 'True'],
    'admin_id': ['0.1', '0', '0', '3', '3'],
    'sex': ['male', 'female', 'male', 'female', 'female'],
    'foreigner': ['0', '0', '0', '0', '0'],
    'birthdate': ['dfs', '1985-03-22', '1992-07-10', '1990-12-01', '1990-12-01'],
    'registed_time': [
        '2023-01-01 10:30:00',
        '2023-01-05 14:20:00',
        '2023-01-10 09:15:00',
        '2023-01-12 11:00:00',
        '2023-01-12 11:00:00'
    ]
}

# ✅ Option 1: via pandas DataFrame
pdf = pd.DataFrame(sample_data)
df = spark.createDataFrame(pdf)


25/11/10 15:42:01 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:
from pathlib import Path
from datetime import datetime
from typing import Tuple, List, Optional, Dict, Any
from pyspark.sql import DataFrame, functions as F, types as T
from pyspark.sql.window import Window

# import project modules
from src.churn_prediction.logger import logger
from src.churn_prediction.pydantic.data_validation_config import DataValidationConfig
from src.churn_prediction.pydantic.pipeline_config import PipelineConfig
from src.churn_prediction.utils.common import load_single_config, get_spark, get_execution_date
from src.churn_prediction.utils.loaders import load_data
from src.churn_prediction.utils.writers import write_data

class DataValidator:
    """
    Data Validator class to perform data quality checks based on configuration.
    """

    def __init__(self, config_path: str, execution_date: Optional[str] = None) -> None:
        """
        Initialize validator with config file.

        Args:
            config_path (str): Path to config YAML file
            execution_date (Optional[str]): Execution date string in 'YYYY-MM-DD' format. Defaults to today's date.
        """
        try:
            logger.info("Initializing DataValidator...")

            self.spark = get_spark()

            # Load config using project's config loader(s)
            self.config_path: Path = Path(config_path)
            self.pipeline_config: PipelineConfig = load_single_config(PipelineConfig, self.config_path)
            self.execution_date: str = get_execution_date(execution_date) if execution_date else datetime.now().strftime("%Y-%m-%d")

            self.validation_path: Path = Path(self.pipeline_config.validation.config_path)
            self.validation_config: DataValidationConfig = load_single_config(DataValidationConfig, self.validation_path)

            self.columns_config = self.validation_config.columns
            self.quality_rules_config = self.validation_config.quality_rules

            self.input_path: str = self.pipeline_config.validation.input.get("file_path").replace("${execution_date}", self.execution_date)
            self.output_path: str = self.pipeline_config.validation.output.get("file_path").replace("${execution_date}", self.execution_date)

        except Exception as e:
            logger.error(f"Error initializing DataValidator: {e}")
            raise

    # Helpers
    @staticmethod
    def map_to_spark_type(type_str: str) -> T.DataType:
        """
        Map config type names to pyspark types.
        Extend mapping if your config has more granular types.
        """
        t = (type_str or "").lower().strip()
        if t in ("string", "str", "varchar", "text"):
            return T.StringType()
        if t in ("integer", "int", "long"):
            return T.LongType()
        if t in ("float", "double", "decimal"):
            return T.DoubleType()
        if t in ("bool", "boolean"):
            return T.BooleanType()
        if t in ("date",):
            return T.DateType()
        if t in ("datetime", "timestamp"):
            return T.TimestampType()
        # fallback
        return T.StringType()

    # Validation methods
    def validate_duplicates_records(self, df: DataFrame) -> Dict[str, Any]:
        """
        Identify duplicate records based on all columns except 'sk_key'.
        Returns dict with 'error' -> list of DataFrames (error reports) and 'clean_df' -> DataFrame with duplicates removed.
        """
        logger.info("Validating duplicate records (excluding sk_key)...")
        cols_to_check = [c for c in df.columns if c != "sk_key"]

        # Count duplicates
        grouped = df.groupBy(*cols_to_check).count()
        duplicates = grouped.filter("count > 1").drop("count")

        if duplicates.rdd.isEmpty():
            # no duplicates
            return {"error": [], "clean_df": df}

        # Join back to fetch duplicate rows
        dup_rows = df.join(duplicates, on=cols_to_check, how="inner") \
            .withColumn("error_type", F.lit("RecordDuplicateViolation")) \
            .withColumn("error_message", F.lit("Duplicate records found based on all columns except 'sk_key'"))

        # Remove duplicates and keep the first occurrence: create row_number over partition and filter row_number == 1
        w = Window.partitionBy(*cols_to_check).orderBy(F.lit(1))
        df_with_rn = df.withColumn("__rn", F.row_number().over(w))
        clean_df = df_with_rn.filter(F.col("__rn") == 1).drop("__rn")

        return {"error": [dup_rows], "clean_df": clean_df}

    def validate_duplicates_keys(self, df: DataFrame) -> Dict[str, Any]:
        """
        Identify duplicate keys based on configured primary key columns.
        """
        logger.info("Validating duplicate key constraints...")
        # Determine key columns from columns_config where primary_keys attribute is truthy.
        key_columns = [c for c, cfg in self.columns_config.items() if getattr(cfg, "primary_keys", False)]

        if not key_columns:
            logger.info("No primary key columns configured; skipping key-duplicate validation.")
            return {"error": [], "clean_df": df}

        grouped = df.groupBy(*key_columns).count()
        dup_keys = grouped.filter("count > 1").drop("count")

        if dup_keys.rdd.isEmpty():
            return {"error": [], "clean_df": df}

        dup_rows = df.join(dup_keys, on=key_columns, how="inner") \
            .withColumn("error_type", F.lit("UniqueKeyDuplicateViolation")) \
            .withColumn("error_message", F.lit(f"Duplicate values found in unique key column(s): {key_columns}"))

        # Remove duplicates
        clean_df = df.join(dup_keys, on=key_columns, how="left_anti")

        return {"error": [dup_rows], "clean_df": clean_df}

    def validate_data_types_and_nullability(self, df: DataFrame) -> Dict[str, Any]:
        """
        Validate data types and nullability:
        - Attempt to cast each column to the target Spark data type.
        - Rows where original is not null but cast yields null are marked as InvalidDataType.
        - Rows where column is null but configured nullable == False are marked as NullableViolation.
        Returns dict with 'error' list (DataFrames) and 'clean_df' (DataFrame with error rows removed).
        """
        logger.info("Validating data types and nullability...")

        errors: List[DataFrame] = []
        working_df = df

        for col_name, cfg in self.columns_config.items():
            expected_type = self.map_to_spark_type(getattr(cfg, "type", "string"))
            nullable = bool(getattr(cfg, "nullable", True))

            # Try cast
            cast_col_name = f"__cast_{col_name}"
            working_df = working_df.withColumn(cast_col_name, F.col(col_name).cast(expected_type))

            # Detect invalid data type: original not null but cast is null (and original not equal to cast when possible)
            invalid_type_mask = (F.col(col_name).isNotNull()) & (F.col(cast_col_name).isNull())
            invalid_rows = working_df.filter(invalid_type_mask).drop(cast_col_name) \
                .withColumn("error_type", F.lit("InvalidDataType")) \
                .withColumn("error_message", F.lit(f"Column '{col_name}' has invalid data type (expected {expected_type.simpleString()})"))

            if not invalid_rows.rdd.isEmpty():
                errors.append(invalid_rows)

            # Detect nullability violation: cast (or original) is null but nullable == False
            if not nullable:
                null_violation_rows = working_df.filter(F.col(col_name).isNull()).drop(cast_col_name) \
                    .withColumn("error_type", F.lit("NullableViolation")) \
                    .withColumn("error_message", F.lit(f"Column '{col_name}' is non-nullable but has null value"))
                if not null_violation_rows.rdd.isEmpty():
                    errors.append(null_violation_rows)

            # Cleanup temporary cast column for subsequent iterations (but keep original column)
            working_df = working_df.drop(cast_col_name)

        # Build clean_df by removing any rows appearing in any error frames.
        if not errors:
            return {"error": [], "clean_df": df}

        # Union all error DataFrames to get IDs of offending rows. To remove offending rows we need a reliable row identifier.
        # If there's a primary key defined use that, otherwise create a synthetic row id.
        key_columns = [c for c, cfg in self.columns_config.items() if getattr(cfg, "primary_keys", False)]
        if not key_columns:
            # create synthetic __row_id
            df_with_id = df.withColumn("__row_id", F.monotonically_increasing_id())
            error_union = None
            for e in errors:
                e_with_id = e.join(df_with_id, on=[c for c in df.columns], how="inner") if df.columns else e
                if error_union is None:
                    error_union = e
                else:
                    error_union = error_union.unionByName(e, allowMissingColumns=True)
            cols_common = [c for c in df.columns if c in error_union.columns]
            clean_df = df.join(error_union.select(*cols_common).distinct(), on=cols_common, how="left_anti")
        else:
            # Use primary keys to identify offending rows
            # Build a union of offending primary key values
            pk_errors = None
            for e in errors:
                pk_part = e.select(*key_columns).distinct()
                pk_errors = pk_part if pk_errors is None else pk_errors.union(pk_part)
            clean_df = df.join(pk_errors.distinct(), on=key_columns, how="left_anti")

        return {"error": errors, "clean_df": clean_df}

    def validate_foreign_keys(self, df: DataFrame) -> Dict[str, Any]:
        """
        Validate foreign key constraints by doing anti-joins against referenced child tables.
        Expects each column config that has foreign_keys to include child_path and child_column.
        """
        logger.info("Validating foreign keys...")
        errors: List[DataFrame] = []
        clean_df = df

        for column, cfg in self.columns_config.items():
            foreign_key = getattr(cfg, "foreign_keys", None)
            if not foreign_key:
                continue

            child_path = foreign_key.child_path
            child_column = foreign_key.child_column

            # Load child table
            try:
                child_df = self.load_data(child_path).select(child_column).distinct()
            except Exception as e:
                logger.warning(f"Failed to load child table for FK check: {child_path}: {e}")
                continue

            # Find invalid rows: left_anti join - rows in parent that don't have matching key in child
            invalid_rows = clean_df.join(child_df, clean_df[column] == child_df[child_column], how="left_anti") \
                .filter(F.col(column).isNotNull()) \
                .withColumn("error_type", F.lit("ForeignKeyViolation")) \
                .withColumn("error_message", F.lit(f"Column '{column}' has values not present in '{child_path}.{child_column}'"))

            if not invalid_rows.rdd.isEmpty():
                errors.append(invalid_rows)
                # Remove invalid rows from clean_df for downstream checks
                # Identify offending PK set if exists, otherwise remove by exact match
                key_columns = [c for c, cfg in self.columns_config.items() if getattr(cfg, "primary_keys", False)]
                if key_columns:
                    # remove by PK
                    invalid_pks = invalid_rows.select(*key_columns).distinct()
                    clean_df = clean_df.join(invalid_pks, on=key_columns, how="left_anti")
                else:
                    # remove rows that match all columns in invalid_rows (may be heavy)
                    cols_common = [c for c in clean_df.columns if c in invalid_rows.columns]
                    clean_df = clean_df.join(invalid_rows.select(*cols_common).distinct(), on=cols_common, how="left_anti")

        return {"error": errors, "clean_df": clean_df}

    # Utility to combine error frames into a single error DataFrame for reporting
    def union_error_dfs(self, errors: List[DataFrame]) -> DataFrame:
        if not errors:
            # Return empty DataFrame with same schema as input if possible
            # As a fallback create a DataFrame with columns: error_type, error_message
            return self.spark.createDataFrame([], schema=T.StructType([
                T.StructField("error_type", T.StringType(), True),
                T.StructField("error_message", T.StringType(), True)
            ]))
        union_df = errors[0]
        for e in errors[1:]:
            union_df = union_df.unionByName(e, allowMissingColumns=True)
        return union_df

    @staticmethod
    def extract_results_error_dfs(result: Dict[str, Any]) -> Tuple[DataFrame, List[DataFrame]]:
        """
        Extract clean_df and list of error DataFrames from a result dict returned by validation helpers.
        """
        clean_df = result.get("clean_df")
        errors = result.get("error", [])
        return clean_df, errors

    # Main validator orchestration
    def validator(self) -> None:
        """
        Run the validation pipeline:
        1. Load data
        2. Drop internal columns
        3. Run duplicate record validation
        4. Run key uniqueness validation
        5. Validate types & nullability
        6. Validate foreign keys
        7. Save cleaned data + errors
        """
        try:
            logger.info("Starting data validation ...")

            all_error_dfs: List[DataFrame] = []

            # 1. Load data
            df = load_data(self.input_path)
            # Drop the dl_load_ts if present (as original code did)
            if "dl_load_ts" in df.columns:
                df = df.drop("dl_load_ts")

            # 2. Validate record duplicates (if configured)
            if not getattr(self.quality_rules_config.allow_record_duplicates, "enabled", False):
                res = self.validate_duplicates_records(df)
                df, errs = self.extract_results_error_dfs(res)
                all_error_dfs.extend(errs)

            # 3. Validate key duplicates
            if not getattr(self.quality_rules_config.allow_key_duplicates, "enabled", False):
                res = self.validate_duplicates_keys(df)
                df, errs = self.extract_results_error_dfs(res)
                all_error_dfs.extend(errs)

            # 4. Validate data types and nullability
            res = self.validate_data_types_and_nullability(df)
            df, errs = self.extract_results_error_dfs(res)
            all_error_dfs.extend(errs)

            # 5. Validate foreign keys
            if not getattr(self.quality_rules_config.foreign_key_checks, "enabled", False):
                res = self.validate_foreign_keys(df)
                df, errs = self.extract_results_error_dfs(res)
                all_error_dfs.extend(errs)

            # 6. Consolidate errors
            if all_error_dfs:
                error_df = self.union_error_dfs(all_error_dfs)
            else:
                # produce empty error dataframe with at least the same columns as df + error columns
                empty_err_schema = T.StructType([T.StructField(c, T.StringType(), True) for c in df.columns])
                empty_err_schema.add(T.StructField("error_type", T.StringType(), True))
                empty_err_schema.add(T.StructField("error_message", T.StringType(), True))
                error_df = self.spark.createDataFrame([], schema=empty_err_schema)

            # 7. Add dl_load_ts timestamps
            now_ts = datetime.now()
            df = df.withColumn("dl_load_ts", F.lit(now_ts))
            error_df = error_df.withColumn("dl_load_ts", F.lit(now_ts))

            # 8. Save results
            # Primary output - cleaned data
            write_data(df, file_path=self.output_path)

            # Error logs saved to a separate path - same as original code replaced 'validated' with 'errors'
            error_output_path = str(self.output_path).replace("validated", "errors")
            write_data(error_df, file_path=error_output_path)

            logger.info("Data validation completed.")

        except Exception as e:
            logger.exception(f"Error during data validation: {e}")
            raise

In [92]:
if __name__ == "__main__":
    config_path = "src/churn_prediction/config/conf/conf_customer_profile.yaml"
    validator = DataValidator(config_path=config_path)
    validator.validator()

[ 2025-11-10 16:04:08 ] | churn_prediction | INFO     | 2012860777.py:__init__:39 | Initializing DataValidator...
[ 2025-11-10 16:04:08 ] | churn_prediction | INFO     | common.py:load_single_config:41 | Successfully loaded schema: src/churn_prediction/config/conf/conf_customer_profile.yaml
[ 2025-11-10 16:04:08 ] | churn_prediction | INFO     | common.py:load_single_config:41 | Successfully loaded schema: src/churn_prediction/config/validation/customer_profile.yaml
[ 2025-11-10 16:04:08 ] | churn_prediction | INFO     | 2012860777.py:validator:295 | Starting data validation (PySpark + Deequ)...
[ 2025-11-10 16:04:08 ] | churn_prediction | INFO     | loaders.py:load_data:170 | Loading data from local: data/raw/customer_profile/2025-11-10/
[ 2025-11-10 16:04:08 ] | churn_prediction | INFO     | 2012860777.py:validate_duplicates_records:90 | Validating duplicate records (excluding sk_key)...
[ 2025-11-10 16:04:09 ] | churn_prediction | INFO     | 2012860777.py:validate_duplicates_keys:11

In [97]:
spark.read.load("data/validated/customer_profile/2025-11-10").show()

+-------+----------+--------------+-----------+--------+-----------+---------+--------------------+-------------------+----------+--------------------+
|user_id|first_name|     last_name|  activated|admin_id|        sex|foreigner|           birthdate|      registed_time|dl_data_dt|          dl_load_ts|
+-------+----------+--------------+-----------+--------+-----------+---------+--------------------+-------------------+----------+--------------------+
| 197713|เทอดศักดิ์|     ทับทิมไทย|       TRUE|       0|       male|        0|1999-07-29 00:00:...|2020-08-18 22:46:56|2025-11-10|2025-11-10 16:04:...|
| 197715|  ปิยะชาติ|     ถนัดอักษร|     REJECT|       0|unspecified|        0|1988-04-15 10:05:...|2020-08-18 23:04:42|2025-11-10|2025-11-10 16:04:...|
| 197717| ฉัตรปรียา|         นกทอง|      FALSE|       0|unspecified|        0|1999-06-07 08:10:...|2020-08-18 23:16:26|2025-11-10|2025-11-10 16:04:...|
| 197719|    มารีนี|ปรีชากุลเศรษฐ์|     REJECT|       0|unspecified|        0|1983-10-02